In [ ]:
!pip install langgraph langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

# Safely retrieve the secret key
groq_api_key = userdata.get('Groq_APIkey')

llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.1-8b-instant",
    temperature=0
)

TimeoutException: Requesting secret Groq_APIkey timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
from typing import TypedDict, List



class ChatState(TypedDict):

    messages: List[str]     # the running conversation

    name: str               # remembered user name

    preferences: str        # remembered preferences

In [ ]:
def chatbot(state: ChatState) -> dict:

    user_msg = state['messages'][-1]



    # very simple memory rules for a beginner demo

    name = state.get('name', '')

    prefs = state.get('preferences', '')

    if 'my name is' in user_msg.lower():

        name = user_msg.lower().split('my name is')[-1].strip().title()

    if 'i like' in user_msg.lower():

        prefs = user_msg.lower().split('i like')[-1].strip()



    context = f"The user's name is {name}. They like {prefs}."

    reply = llm.invoke(context + ' Reply to: ' + user_msg).content



    return {'messages': state['messages'] + [reply],

            'name': name, 'preferences': prefs}

In [ ]:
from langgraph.graph import StateGraph, START, END



builder = StateGraph(ChatState)

builder.add_node('chatbot', chatbot)

builder.add_edge(START, 'chatbot')

builder.add_edge('chatbot', END)

graph = builder.compile()

In [ ]:
state = {'messages': [], 'name': '', 'preferences': ''}



while True:

    user = input('You: ')

    if user.lower() == 'quit':

        break

    state['messages'].append(user)

    state = graph.invoke(state)

    print('Bot:', state['messages'][-1])

You: HELLO


NameError: name 'graph' is not defined